In [33]:
!pip install bertviz

In [34]:
from transformers import AutoTokenizer

In [35]:
from bertviz.transformers_neuron_view import BertModel
from bertviz.neuron_view import show

In [36]:
model_ckpt = 'bert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)
model = BertModel.from_pretrained(model_ckpt)

In [37]:
model

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): BertLayerNorm()
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): BertLayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
        )
        (intermediate): BertIntermediate(
          (den

In [38]:
sample_text = 'time flies like an arrow'
show(model,model_type='bert',tokenizer=tokenizer,sentence_a=sample_text,display_mode='light',layer=0,head=8)

Output hidden; open in https://colab.research.google.com to view.

# 自注意力计算

In [39]:
tokenizer.model_input_names

['input_ids', 'token_type_ids', 'attention_mask']

In [40]:
model_inputs = tokenizer(sample_text,return_tensors='pt',add_special_tokens=False)
model_inputs

{'input_ids': tensor([[ 2051, 10029,  2066,  2019,  8612]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1]])}

In [41]:
from torch import nn
import numpy as np
import torch
from transformers import AutoConfig

In [42]:
config = AutoConfig.from_pretrained(model_ckpt)
config

BertConfig {
  "add_cross_attention": false,
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": null,
  "classifier_dropout": null,
  "eos_token_id": null,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "is_decoder": false,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "tie_word_embeddings": true,
  "transformers_version": "5.12.1",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 30522
}

In [43]:
token_embedding = nn.Embedding(config.vocab_size,config.hidden_size)
token_embedding

Embedding(30522, 768)

In [44]:
input_embeddings = token_embedding(model_inputs['input_ids'])
input_embeddings.shape

torch.Size([1, 5, 768])

In [45]:
query = key =value = input_embeddings
# 获取张量k的最后一个维度
dim_k = key.size(-1)
print('dim_k',dim_k)

dim_k 768


In [46]:
# bmm是矩阵批量乘法，transpose为转置，此处转置1 2维，sqrt是平方根
attn_scores = torch.bmm(query,key.transpose(1,2)) / np.sqrt(dim_k)
print('attn_scores.shape:',attn_scores.shape)
attn_scores

attn_scores.shape: torch.Size([1, 5, 5])


tensor([[[27.5216, -1.2226, -0.3822,  0.4682, -0.1810],
         [-1.2226, 26.6071, -1.1571, -0.1056, -0.0537],
         [-0.3822, -1.1571, 28.8722, -0.2371, -1.5662],
         [ 0.4682, -0.1056, -0.2371, 27.3538,  1.6988],
         [-0.1810, -0.0537, -1.5662,  1.6988, 28.3026]]],
       grad_fn=<DivBackward0>)

In [47]:
import torch.nn.functional as F
attn_weights = F.softmax(attn_scores,dim=-1)
attn_weights

tensor([[[1.0000e+00, 3.2851e-13, 7.6120e-13, 1.7817e-12, 9.3090e-13],
         [8.1984e-13, 1.0000e+00, 8.7526e-13, 2.5051e-12, 2.6385e-12],
         [1.9722e-13, 9.0869e-14, 1.0000e+00, 2.2802e-13, 6.0363e-14],
         [2.1074e-12, 1.1873e-12, 1.0409e-12, 1.0000e+00, 7.2140e-12],
         [4.2633e-13, 4.8420e-13, 1.0670e-13, 2.7933e-12, 1.0000e+00]]],
       grad_fn=<SoftmaxBackward0>)

In [48]:
attn_weights.sum(axis=-1)

tensor([[1., 1., 1., 1., 1.]], grad_fn=<SumBackward1>)

In [49]:
attn_outputs = torch.bmm(attn_weights,value)
attn_outputs

tensor([[[-0.1997,  0.3822,  0.3294,  ...,  1.4127, -1.4132,  0.4602],
         [ 0.5653, -0.1200,  0.6723,  ...,  0.3539, -1.4047,  1.1380],
         [-0.2164, -0.1675, -0.6473,  ...,  0.4922,  1.3867,  1.9322],
         [ 0.4475,  0.6281, -0.7153,  ...,  0.3462,  0.0508,  0.2188],
         [-0.5683, -0.2894,  0.0391,  ..., -0.1367,  2.2065,  0.3888]]],
       grad_fn=<BmmBackward0>)

In [50]:
def scaled_dot_attention(query,key,value):
  dim_k = key.size(-1)
  attn_scores = torch.bmm(query,key.transpose(1,2) / np.sqrt(dim_k))
  attn_weights = F.softmax(attn_scores,dim=-1)
  return torch.bmm(attn_weights,value)

In [51]:
scaled_dot_attention(query,key,value)

tensor([[[-0.1997,  0.3822,  0.3294,  ...,  1.4127, -1.4132,  0.4602],
         [ 0.5653, -0.1200,  0.6723,  ...,  0.3539, -1.4047,  1.1380],
         [-0.2164, -0.1675, -0.6473,  ...,  0.4922,  1.3867,  1.9322],
         [ 0.4475,  0.6281, -0.7153,  ...,  0.3462,  0.0508,  0.2188],
         [-0.5683, -0.2894,  0.0391,  ..., -0.1367,  2.2065,  0.3888]]],
       grad_fn=<BmmBackward0>)

# 注意力头与多头注意力

In [52]:
class AttentionHead(nn.Module):
  def __init__(self,embed_dim,head_dim):
    super().__init__()
    self.Wq = nn.Linear(embed_dim,head_dim)
    self.Wk = nn.Linear(embed_dim,head_dim)
    self.Wv = nn.Linear(embed_dim,head_dim)
  def forward(self,hidden_states):
    q = self.Wq(hidden_states)
    k = self.Wk(hidden_states)
    v = self.Wv(hidden_states)
    attn_outputs = scaled_dot_attention(q,k,v)
    return attn_outputs

In [58]:
class MultiHeadAttention(nn.Module):
  def __init__(self,config):
    super().__init__()
    embed_dim = config.hidden_size
    num_heads = config.num_attention_heads

    head_dim = embed_dim // num_heads
    self.heads = nn.ModuleList({
        AttentionHead(embed_dim,head_dim) for _ in range(num_heads)
    })
    self.output_layer = nn.Linear(embed_dim,embed_dim)
  def forward(self,hidden_state):
    print(f'input hidden_state:{hidden_state.shape}')
    print(f'head(hidden_state):{self.heads[11](hidden_state).shape}')
    x = torch.cat([head(hidden_state) for head in self.heads],dim=-1)
    print(f'x.shape:{x.shape}')
    x = self.output_layer(x)
    return x

In [54]:
config

BertConfig {
  "add_cross_attention": false,
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": null,
  "classifier_dropout": null,
  "eos_token_id": null,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "is_decoder": false,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "tie_word_embeddings": true,
  "transformers_version": "5.12.1",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 30522
}

In [59]:
mha = MultiHeadAttention(config)

In [60]:
token_embedding
input_embeddings
input_embeddings.shape

torch.Size([1, 5, 768])

In [61]:
mha(input_embeddings)

input hidden_state:torch.Size([1, 5, 768])
head(hidden_state):torch.Size([1, 5, 64])
x.shape:torch.Size([1, 5, 768])


tensor([[[-0.2900,  0.0656,  0.0576,  ..., -0.0193, -0.0028,  0.2036],
         [-0.2181,  0.2011, -0.0253,  ...,  0.0642, -0.0472,  0.1860],
         [-0.3086,  0.1537, -0.0826,  ...,  0.0016, -0.0217,  0.2623],
         [-0.2560,  0.1336,  0.1723,  ...,  0.0618, -0.0078,  0.1880],
         [-0.3431,  0.1116,  0.1293,  ...,  0.0769,  0.0777,  0.2170]]],
       grad_fn=<ViewBackward0>)

In [62]:
model

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): BertLayerNorm()
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): BertLayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
        )
        (intermediate): BertIntermediate(
          (den

In [63]:
sentence_a = 'time flies like an arrow'
sentence_b = 'fruit flies like an banana'
viz_inputs = tokenizer(sentence_a,sentence_b,return_tensors='pt')

In [64]:
print(viz_inputs)

{'input_ids': tensor([[  101,  2051, 10029,  2066,  2019,  8612,   102,  5909, 10029,  2066,
          2019, 15212,   102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}


In [65]:
attention = model(**viz_inputs).attentions
sentence_b_start = (viz_inputs.token_type_ids == 0).sum(dim=1)
print(sentence_b_start)
tokens = tokenizer.convert_ids_to_tokens(viz_inputs.input_ids[0])
head_view(attention,tokens,sentence_b_start,head=[8])

AttributeError: 'tuple' object has no attribute 'attentions'